# 个人投资分析助手

使用流程：
1. **读公众号文章** — 粘贴链接，自动提取内容和图片
2. **输入指数估值** — 手动输入或查看参考数据
3. **分析股票/基金** — 输入代码，自动拉取数据 + 估值分析
4. **生成报告** — 一键输出分析总结

---
## Part 0: 安装依赖（首次使用运行一次）

In [ ]:
!pip install akshare beautifulsoup4 lxml Pillow plotly -q

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from article_reader import fetch_article, download_images, extract_stock_codes
from market_data import (
    get_stock_history, get_stock_info, get_fund_nav,
    get_fund_info, get_index_valuation,
)
from valuation import (
    analyze_stock, analyze_fund, compare_valuation,
    generate_report, plot_stock_chart, plot_fund_chart,
)
from IPython.display import display, Image, Markdown, HTML
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print('导入完成 ✓')

---
## Part 1: 读取公众号文章

把微信公众号文章链接粘贴到下方，运行后会自动提取：
- 文章标题、作者、发布时间
- 文章正文（纯文本）
- 文章中的所有图片

In [ ]:
# ============ 在这里粘贴公众号文章链接 ============
article_url = ""
# ================================================

if article_url:
    article = fetch_article(article_url)
    print(f"标题: {article['title']}")
    print(f"作者: {article['author']}")
    print(f"时间: {article['publish_time']}")
    print(f"图片数: {article['image_count']}")
    print(f"\n{'='*60}")
    print("正文预览（前2000字）:")
    print(article['content_text'][:2000])
else:
    print('请先粘贴公众号文章链接到 article_url 变量')

### 下载并显示文章图片

运行下面的 cell 会下载文章中的图片到 `output/images/` 目录，并在 Notebook 中显示。

In [ ]:
if article_url and article.get('images'):
    print(f"正在下载 {article['image_count']} 张图片...")
    local_images = download_images(article['images'])
    print(f"下载完成，保存到 output/images/\n")
    
    # 在 Notebook 中显示前 10 张图片
    for i, path in enumerate(local_images[:10]):
        print(f"图片 {i+1}: {path}")
        display(Image(filename=path, width=600))
else:
    print('没有图片可下载（请先运行上面的 cell 抓取文章）')

### 提取文章中提到的股票/基金代码

自动从文章正文中识别 6 位数字代码（股票/基金代码）。

In [ ]:
if article_url and article.get('content_text'):
    codes = extract_stock_codes(article['content_text'])
    if codes:
        print(f"从文章中提取到 {len(codes)} 个可能的代码:")
        for c in codes:
            print(f"  {c}")
    else:
        print('未从文章中提取到股票/基金代码')
else:
    print('请先运行 Part 1 抓取文章')

---
## Part 2: 输入指数估值数据

你可以：
- **方式一**：手动输入你已知的指数估值数据（比如从公众号文章、截图中看到的）
- **方式二**：运行下方代码自动获取（需要 akshare 支持该指数）

In [ ]:
# ============ 方式一：手动输入指数估值 ============
# 修改下面的值为你从公众号/截图中获取的数据

user_index_valuation = {
    "index_name": "沪深300",
    "pe": None,              # 如: 11.5
    "pb": None,              # 如: 1.3
    "pe_percentile": None,   # 如: 0.25 (25%)
    "pb_percentile": None,   # 如: 0.20 (20%)
    "dividend_yield": None,  # 如: 3.2
}

# 检查是否已填写
filled = {k: v for k, v in user_index_valuation.items() if v is not None and k != 'index_name'}
if filled:
    print('已填写的指数估值数据:')
    for k, v in filled.items():
        print(f'  {k}: {v}')
else:
    print('尚未填写指数估值数据，请修改上方 cell 中的值')

In [ ]:
# ============ 方式二：自动获取指数估值 ============
# 取消注释并运行，会自动从 akshare 获取

# 沪深300
# auto_val = get_index_valuation("000300")
# print("沪深300 估值:")
# for k, v in auto_val.items():
#     print(f"  {k}: {v}")

# 中证500
# auto_val_500 = get_index_valuation("000905")
# print("\n中证500 估值:")
# for k, v in auto_val_500.items():
#     print(f"  {k}: {v}")

print('提示: 取消上方注释并运行，可自动获取指数估值')

### 上传/显示估值截图

如果你有估值数据的截图，可以把图片文件放到 `output/images/` 目录，然后在下方显示。

In [ ]:
# 显示本地图片（修改路径为你的图片文件）
# image_path = "output/images/your_valuation_screenshot.png"
# if os.path.exists(image_path):
#     display(Image(filename=image_path, width=800))
# else:
#     print(f'图片不存在: {image_path}')

print('提示: 把估值截图放到 output/images/ 目录，修改上方路径后运行即可显示')

---
## Part 3: 分析个股

输入股票代码，自动获取行情、估值、基本面数据，并给出投资建议。

In [ ]:
# ============ 在这里输入要分析的股票代码 ============
stock_symbol = "600519"  # 贵州茅台
# ==================================================

if stock_symbol:
    print(f"正在分析 {stock_symbol} ...\n")
    result = analyze_stock(stock_symbol, user_valuation=user_index_valuation)
    
    print(f"股票: {result['name']} ({result['code']})")
    print(f"建议: {result['recommendation']}")
    print(f"评分: {result['score']}/100 (越低越值得买)")
    print(f"理由: {result['reason']}")
    
    # 基本信息
    info = result['basic_info']
    if info:
        print(f"\n基本信息:")
        for k in ['pe', 'pb', 'market_cap', 'circulating_cap']:
            v = info.get(k)
            if v is not None:
                label = {'pe': '市盈率', 'pb': '市净率', 'market_cap': '总市值(亿)', 'circulating_cap': '流通市值(亿)'}[k]
                print(f"  {label}: {v}")
    
    # 价格统计
    ps = result['price_stats']
    if ps:
        print(f"\n价格统计 (近{ps.get('data_days', '-')}个交易日):")
        print(f"  最新价: {ps['latest_price']}")
        print(f"  52周最高: {ps['high_52w']} (距高点 {ps['price_vs_high']}%)")
        print(f"  52周最低: {ps['low_52w']} (距低点 +{ps['price_vs_low']}%)")
        print(f"  20日涨跌: {ps['change_pct_20d']}%")
else:
    print('请先输入股票代码')

### 股票 K 线图

In [ ]:
if stock_symbol:
    fig = plot_stock_chart(stock_symbol, days=180)
    fig.show()
else:
    print('请先输入股票代码')

---
## Part 4: 分析基金

输入基金代码，自动获取净值和基金信息，并给出投资建议。

In [ ]:
# ============ 在这里输入要分析的基金代码 ============
fund_code = "110011"  # 易方达中小盘
# ==================================================

if fund_code:
    print(f"正在分析基金 {fund_code} ...\n")
    result = analyze_fund(fund_code, user_valuation=user_index_valuation)
    
    print(f"基金: {result['name']} ({result['code']})")
    print(f"建议: {result['recommendation']}")
    print(f"评分: {result['score']}/100 (越低越值得买)")
    print(f"理由: {result['reason']}")
    
    info = result['basic_info']
    if info:
        if info.get('fund_type'):
            print(f"类型: {info['fund_type']}")
        if info.get('nav'):
            print(f"最新净值: {info['nav']} ({info.get('nav_date', '')})")
    
    ps = result['price_stats']
    if ps:
        print(f"\n净值统计:")
        print(f"  最新净值: {ps['latest_nav']}")
        print(f"  历史最高: {ps['high']} (距高点 {ps['nav_vs_high']}%)")
        print(f"  历史最低: {ps['low']} (距低点 +{ps['nav_vs_low']}%)")
else:
    print('请先输入基金代码')

### 基金净值曲线

In [ ]:
if fund_code:
    fig = plot_fund_chart(fund_code)
    fig.show()
else:
    print('请先输入基金代码')

---
## Part 5: 批量分析

一次分析多只股票/基金，生成汇总表格和报告。

In [ ]:
# ============ 在这里输入要批量分析的标的 ============
# 格式: (代码, 类型) 类型为 "stock" 或 "fund"
targets = [
    ("600519", "stock"),   # 贵州茅台
    ("000858", "stock"),   # 五粮液
    # ("110011", "fund"),  # 易方达中小盘
    # ("000300", "index"), # 沪深300 (作为参考)
]
# ==================================================

all_results = []

for code, kind in targets:
    print(f"分析 {kind}: {code} ...")
    try:
        if kind == "stock":
            r = analyze_stock(code, user_valuation=user_index_valuation)
        elif kind == "fund":
            r = analyze_fund(code, user_valuation=user_index_valuation)
        else:
            continue
        all_results.append(r)
    except Exception as e:
        print(f"  失败: {e}")

print(f"\n分析完成，共 {len(all_results)} 个标的")

In [ ]:
# 显示汇总表格
if all_results:
    rows = []
    for r in all_results:
        v = r.get('valuation', {})
        rows.append({
            '标的': r['name'],
            '代码': r['code'],
            'PE': f"{v['pe']:.1f}" if v.get('pe') else '-',
            'PB': f"{v['pb']:.1f}" if v.get('pb') else '-',
            '建议': r['recommendation'],
            '评分': r['score'],
            '理由': r['reason'],
        })
    df = pd.DataFrame(rows)
    display(df)
else:
    print('请先运行上面的批量分析')

---
## Part 6: 生成报告

In [ ]:
if all_results:
    report = generate_report(all_results, index_valuation=user_index_valuation)
    display(Markdown(report))
    
    # 保存到文件
    report_path = "output/analysis_report.md"
    os.makedirs("output", exist_ok=True)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"\n报告已保存到: {report_path}")
else:
    print('请先运行 Part 5 的批量分析')

---
## 快速入口

以下是一些常用的操作快捷方式：

In [ ]:
# 快速分析：一行代码分析一只股票
# r = analyze_stock("600519", user_valuation=user_index_valuation)
# print(f"{r['name']}: {r['recommendation']} (评分 {r['score']}) — {r['reason']}")

# 快速看图
# plot_stock_chart("600519").show()
# plot_fund_chart("110011").show()

# 快速获取指数估值
# val = get_index_valuation("000300")
# print(val)

print('取消上方注释并运行即可使用快捷方式')